## Import 

In [1]:
import math
import pickle 
import warnings
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import interp
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
from tableone import TableOne

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
path_data   = "../Data/EHR/"
path_cohort = "../Data/Cohorts/"
path_mimic_2008 = "PATH TO DATA/mimic-iii(carevue.1.4)/"
race_path   = "Extraction/MIMICIII/Data/csvExtract/"

### Read Cohorts Splits

In [3]:
with open(path_cohort + "short_icustays", "rb") as fp:   
    short_icustays = pickle.load(fp)
    
with open(path_cohort + "train_icustays", "rb") as fp:   
    train_icustays = pickle.load(fp)
    
with open(path_cohort + "valid_icustays", "rb") as fp:   
    valid_icustays = pickle.load(fp)
    
with open(path_cohort + "test_icustays", "rb") as fp:   
    test_icustays = pickle.load(fp)
    
with open(path_cohort + "icustays_ehr_text", "rb") as fp:   
    icustays_ehr_text = pickle.load(fp)
    
with open(path_cohort + "icustays_ehr_only", "rb") as fp:   
    icustays_ehr_only = pickle.load(fp)

### Reading Data

In [4]:
df_ehr = pd.read_csv(path_data + '0h_to_24h_data.csv', low_memory=False, index_col=False)
df_ehr.head(2)

In [5]:
print(df_ehr.ICUSTAY_ID.nunique())
print(df_ehr.shape)

59653
(1364554, 543)


### Drop Repeated Rows & Keep Notes

In [6]:
df_note = df_ehr[['ICUSTAY_ID', 'Note']]
df_note = df_note[df_note.Note.notnull()]
df_note = df_note.drop_duplicates()

In [7]:
all_columns = list(df_ehr.columns)
remove_columns = [col for col in all_columns if ('_tslm' in col) or ('_diff' in col)]

text_columns = ['Note', 'Discharge_Note', 'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator']

remove_columns.extend(text_columns)
df_ehr.drop(remove_columns, axis=1, inplace=True)
df_ehr = df_ehr.drop_duplicates()

print(df_ehr.ICUSTAY_ID.nunique())
print(df_ehr.shape)

### Fix Age

In [11]:
df_ehr.loc[df_ehr['AGE'] >= 95, 'AGE'] = 95
df_ehr = df_ehr[df_ehr.AGE > 16]

### Take first hours of ICU of patients with more than 24 hour LoS

In [12]:
max_rows = df_ehr.groupby('ICUSTAY_ID').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [13]:
def take_observation_window(df, observation_window):
    
    df = df.groupby('ICUSTAY_ID').head(observation_window).reset_index(drop=True)
    
    return df

In [14]:
df_ehr = take_observation_window(df_ehr, observation_window)

### Remove Short Stay 

In [16]:
split_label = 'HOSPITAL_EXPIRE_FLAG'
label = 'ICU_EXPIRE_FLAG'

In [15]:
df = df_ehr[~df_ehr.ICUSTAY_ID.isin(short_icustays)].copy()

### Variables Selection

In [22]:
selected_columns = ['SUBJECT_ID', 'ICUSTAY_ID', 'Heart Rate', 'SpO2', 'Oxygen Saturation', 'Respiratory Rate', 
                    'Temperature', 'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
                    'Non Invasive Blood Pressure diastolic', 'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
                    'Anion Gap', 'Bicarbonate', 'Lactate', 'Hemoglobin', 'Hematocrit', 'pH', 'Bilirubin, Direct',
                    'pO2', 'pCO2', 'AST', 'ALT', 'WBC', 'RBC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
                    'Phosphate', 'FiO2', 'PEEP', 'Tidal Volume', 'UrineOutput_IO', 
                    'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                    'GCS Total', 'Richmond-RAS Scale',
                    'AGE', 'GENDER', 'ETHNICITY',  
                    'ICU_LOS_H', 'ICU_EXPIRE_FLAG', 'HOSPITAL_EXPIRE_FLAG']

In [23]:
for col in selected_columns:
    temp_col = col + '_ind'
    
    if temp_col in list(df.columns):
        df.loc[df[temp_col] == 0, col] = np.nan

In [24]:
df = df[selected_columns]

In [25]:
df.head(3)

In [26]:
df_ehr = df.drop(['SUBJECT_ID', 'ICUSTAY_ID', 'AGE', 'GENDER', 'ETHNICITY', 'ICU_LOS_H', 'HOSPITAL_EXPIRE_FLAG'], axis=1)
df_demog = df[['SUBJECT_ID', 'ICUSTAY_ID', 'AGE', 'GENDER', 'ETHNICITY', 'ICU_LOS_H', 'ICU_EXPIRE_FLAG']]

### Create TableOne

In [27]:
columns = [ 'Heart Rate', 'SpO2', 'Oxygen Saturation', 'Respiratory Rate', 
            'Temperature', 'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
            'Non Invasive Blood Pressure diastolic', 'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
            'Anion Gap', 'Bicarbonate', 'Lactate', 'Hemoglobin', 'Hematocrit', 'pH', 'Bilirubin, Direct',
            'pO2', 'pCO2', 'AST', 'ALT', 'WBC', 'RBC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
            'Phosphate', 'FiO2', 'PEEP', 'Tidal Volume', 'UrineOutput_IO', 
            'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
            'GCS Total', 'Richmond-RAS Scale',
            'ICU_EXPIRE_FLAG']

In [28]:
categorical = ['Richmond-RAS Scale']

In [29]:
IQR = [ 'Heart Rate', 'SpO2', 'Oxygen Saturation', 'Respiratory Rate', 
        'Temperature', 'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
        'Non Invasive Blood Pressure diastolic', 'Glucose', 'Creatinine', 'Base Excess', 'BUN', 
        'Anion Gap', 'Bicarbonate', 'Lactate', 'Hemoglobin', 'Hematocrit', 'pH', 'Bilirubin, Direct',
        'pO2', 'pCO2', 'AST', 'ALT', 'WBC', 'RBC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
        'Phosphate', 'FiO2', 'PEEP', 'Tidal Volume', 'UrineOutput_IO', 
        'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS', 
        'GCS Total']

In [30]:
EHR_table = TableOne(df_ehr, groupby='ICU_EXPIRE_FLAG', columns=columns, categorical=categorical, pval=True, nonnormal=IQR)

In [31]:
EHR_table

Grouped by ICU_EXPIRE_FLAG                                                                       
                                                                              Missing              Overall                  0.0                  1.0 P-Value
n                                                                                                  1127592              1045848                81744        
Heart Rate, median [Q1,Q3]                                                     109738     84.0 [73.0,97.0]     84.0 [72.7,96.0]    90.0 [76.0,105.0]  <0.001
SpO2, median [Q1,Q3]                                                           144037     98.0 [96.0,99.8]     98.0 [96.0,99.7]    98.0 [95.0,100.0]  <0.001
Oxygen Saturation, median [Q1,Q3]                                             1072343     96.0 [84.5,98.0]     97.0 [85.0,98.0]     95.0 [82.0,98.0]  <0.001
Respiratory Rate, median [Q1,Q3]                                               142725     18.0 [15.0,22.0]     18.0 [15.0,22.0]     20.0 [16.0,25.0]  <0.001
Temperature, median [Q1,Q3]                                                    736665     36.9 [36.4,37.5]     36.9 [36.4,37.5]     36.8 [36.1,37.5]  <0.001
Non Invasive Blood Pressure mean, median [Q1,Q3]                               498920     75.0 [66.0,86.0]     75.3 [66.0,86.0]     70.0 [61.9,81.0]  <0.001
Non Invasive Blood Pressure systolic, median [Q1,Q3]                           495951  116.0 [103.0,133.0]  117.0 [103.8,133.0]   109.0 [96.0,126.0]  <0.001
Non Invasive Blood Pressure diastolic, median [Q1,Q3]                          496169     60.0 [50.0,70.0]     60.0 [51.0,70.0]     55.0 [46.0,66.0]  <0.001
Glucose, median [Q1,Q3]                                                        815218  129.0 [106.0,162.0]  128.5 [106.0,160.5]  140.0 [109.0,185.0]  <0.001
Creatinine, median [Q1,Q3]                                                    1008551        1.0 [0.7,1.6]        1.0 [0.7,1.5]        1.4 [0.9,2.4]  <0.001
Base Excess, median [Q1,Q3]                                                    986447      -1.0 [-4.0,1.0]       0.0 [-3.0,1.0]      -3.0 [-8.0,0.0]  <0.001
BUN, median [Q1,Q3]                                                           1009067     20.0 [13.0,34.0]     19.0 [13.0,32.0]     32.0 [19.0,52.0]  <0.001
Anion Gap, median [Q1,Q3]                                                     1016158     14.0 [11.0,16.0]     13.0 [11.0,16.0]     16.0 [13.0,19.0]  <0.001
Bicarbonate, median [Q1,Q3]                                                   1009419     24.0 [21.0,26.0]     24.0 [21.0,26.0]     21.0 [17.0,25.0]  <0.001
Lactate, median [Q1,Q3]                                                       1059081        2.0 [1.3,3.2]        1.9 [1.3,2.9]        3.0 [1.8,5.4]  <0.001
Hemoglobin, median [Q1,Q3]                                                     957478      10.2 [9.1,11.5]      10.2 [9.1,11.5]      10.2 [9.0,11.5]   0.001
Hematocrit, median [Q1,Q3]                                                     957478     30.3 [27.1,34.0]     30.3 [27.1,34.0]     30.4 [27.0,34.5]   0.001
pH, median [Q1,Q3]                                                             965190        7.4 [7.3,7.4]        7.4 [7.3,7.4]        7.3 [7.2,7.4]  <0.001
Bilirubin, Direct, median [Q1,Q3]                                             1097336        0.4 [0.1,1.3]        0.4 [0.1,1.2]        0.7 [0.2,2.6]  <0.001
pO2, median [Q1,Q3]                                                            985373   129.0 [92.0,195.0]   131.0 [93.0,199.0]   111.0 [80.0,165.0]  <0.001
pCO2, median [Q1,Q3]                                                           985362     40.0 [36.0,46.0]     40.0 [36.0,46.0]     38.5 [33.0,46.0]  <0.001
AST, median [Q1,Q3]                                                           1099379    47.0 [25.0,111.0]    44.0 [25.0,104.0]    70.0 [33.0,169.0]  <0.001
ALT, median [Q1,Q3]                                                           1098963     34.0 [18.0,82.0]     33.0 [18.0,78.0]    43.

### Number of Notes

In [32]:
df_note = df_note[df_note.Note.notna()]
df_note = df_note.groupby('ICUSTAY_ID').count().reset_index()

In [33]:
df_note.head(3)

### Static Information

In [34]:
df_demog = df_demog.groupby('ICUSTAY_ID').head(1)
df_demog = df_demog.merge(df_note, on='ICUSTAY_ID', how='left')

In [35]:
df_demog.head()

### Read Race Dictionary

In [36]:
with open(race_path + 'race_dictionary.pkl', 'rb') as f:
    race_dictionary = pickle.load(f)

In [37]:
general_ethnicity_mapping = {
    
    'WHITE': 'White',
    'WHITE - RUSSIAN': 'White',
    'WHITE - BRAZILIAN': 'White',
    'WHITE - OTHER EUROPEAN': 'White',
    'WHITE - EASTERN EUROPEAN': 'White',
    'PORTUGUESE': 'White',
    
    'UNABLE TO OBTAIN': 'Unknown',
    'UNKNOWN/NOT SPECIFIED': 'Unknown',
    'PATIENT DECLINED TO ANSWER': 'Unknown',
    
    'OTHER': 'Other',
    'MIDDLE EASTERN': 'Other',
    'CARIBBEAN ISLAND': 'Other',
    'MULTI RACE ETHNICITY': 'Other',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Other',
    
    'ASIAN': 'Asian',
    'ASIAN - THAI': 'Asian',
    'ASIAN - OTHER': 'Asian',
    'ASIAN - KOREAN': 'Asian',
    'ASIAN - CHINESE': 'Asian',
    'ASIAN - FILIPINO': 'Asian',
    'ASIAN - JAPANESE': 'Asian',
    'ASIAN - CAMBODIAN': 'Asian',
    'ASIAN - VIETNAMESE': 'Asian',
    'ASIAN - ASIAN INDIAN': 'Asian',
    
    'AMERICAN INDIAN/ALASKA NATIVE': 'Native American',
    'AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE': 'Native American',
    
    'BLACK/AFRICAN': 'Black/African American',
    'BLACK/HAITIAN': 'Black/African American',
    'BLACK/CAPE VERDEAN': 'Black/African American',
    'BLACK/AFRICAN AMERICAN': 'Black/African American',
    
    'SOUTH AMERICAN': 'Hispanic/Latino',
    'HISPANIC OR LATINO': 'Hispanic/Latino',
    'HISPANIC/LATINO - CUBAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - MEXICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - HONDURAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - DOMINICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - COLOMBIAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - SALVADORAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - GUATEMALAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - PUERTO RICAN': 'Hispanic/Latino',
    'HISPANIC/LATINO - CENTRAL AMERICAN (OTHER)': 'Hispanic/Latino'}

In [38]:
def replace_ethnicity_with_names(df, ethnicity_dict):
    
    inv_ethnicity_dict = {v: k for k, v in ethnicity_dict.items()}
    df['ETHNICITY'] = df['ETHNICITY'].map(inv_ethnicity_dict)
    
    return df

In [39]:
def categorize_ethnicity(df, new_mapping):
    
    df['ETHNICITY'] = df['ETHNICITY'].map(new_mapping)
    
    return df

In [40]:
df_demog = replace_ethnicity_with_names(df_demog, race_dictionary)
df_demog = categorize_ethnicity(df_demog, general_ethnicity_mapping)

In [41]:
df_demog = df_demog.rename(columns={"ETHNICITY": "RACE"})
df_demog['ETHNICITY'] = 'Non Hispanic'
df_demog.loc[df_demog.RACE == 'Hispanic/Latino', 'ETHNICITY'] = 'Hispanic'

In [42]:
df_demog.head()

In [43]:
print(df_demog.SUBJECT_ID.nunique())
print(df_demog[df_demog.ICU_EXPIRE_FLAG == 0].SUBJECT_ID.nunique())
print(df_demog[df_demog.ICU_EXPIRE_FLAG == 1].SUBJECT_ID.nunique())

34698
32319
3406


In [44]:
columns = ['AGE', 'GENDER', 'RACE', 'ETHNICITY', 'ICU_LOS_H', 'ICU_EXPIRE_FLAG', 'Note']

categorical = ['GENDER', 'RACE', 'ETHNICITY']

IQR = ['AGE', 'ICU_LOS_H', 'Note']

In [45]:
demog_table = TableOne(df_demog, groupby='ICU_EXPIRE_FLAG', columns=columns, categorical=categorical, pval=True, nonnormal=IQR)

In [46]:
demog_table

Grouped by ICU_EXPIRE_FLAG                                                                  
                                                                    Missing            Overall                0.0                 1.0 P-Value
n                                                                                        46983              43577                3406        
AGE, median [Q1,Q3]                                                       0   66.0 [53.0,78.0]   65.0 [53.0,77.0]    73.0 [60.0,82.0]  <0.001
GENDER, n (%)             1.0                                             0       26603 (56.6)       24738 (56.8)         1865 (54.8)   0.024
                          2.0                                                     20380 (43.4)       18839 (43.2)         1541 (45.2)        
RACE, n (%)               Asian                                           0         1086 (2.3)          994 (2.3)            92 (2.7)  <0.001
                          Black/African American                                    4440 (9.5)         4196 (9.6)           244 (7.2)        
                          Hispanic/Latino                                           1583 (3.4)         1510 (3.5)            73 (2.1)        
                          Native American                                             23 (0.0)           20 (0.0)             3 (0.1)        
                          Other                                                     1190 (2.5)         1109 (2.5)            81 (2.4)        
                          Unknown                                                  4882 (10.4)        4343 (10.0)          539 (15.8)        
                          White                                                   33779 (71.9)       31405 (72.1)         2374 (69.7)        
ETHNICITY, n (%)          Hispanic                                        0         1583 (3.4)         1510 (3.5)            73 (2.1)  <0.001
                          Non Hispanic                                            45400 (96.6)       42067 (96.5)         3333 (97.9)        
ICU_LOS_H, median [Q1,Q3]                                                 0  56.7 [34.6,109.8]  54.5 [33.7,100.8]  114.0 [52.3,235.2]  <0.001
Note, median [Q1,Q3]                                                   2880      4.0 [3.0,6.0]      4.0 [3.0,6.0]       6.0 [4.0,8.0]  <0.001
[1] Chi-squared tests for the following variables may be invalid due to the low number of observations: RACE.

### Save Tables

In [47]:
# EHR_table.to_csv('./Results/TableONe_EHR_ICU_Mortality.csv')
# demog_table.to_csv('./Results/TableONe_DEMOG_ICU_Mortality.csv')